In [ ]:
# ==============================================================================
# ML PRACTICAL 3: BANK CUSTOMER CHURN PREDICTION
# Objective: Predict customer churn using Artificial Neural Networks (ANN)
# ==============================================================================

# 1. Import Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input

# 2. Load Dataset
df = pd.read_csv("Churn_Modelling.csv")
print(f"Dataset Shape: {df.shape}")
print("\nFirst 5 rows:")
print(df.head())

# 3. Data Cleaning & Encoding
# Drop irrelevant identifier columns
df = df.drop(columns=['RowNumber', 'CustomerId', 'Surname'], errors='ignore')
print(f"\nMissing Values: {df.isnull().sum().sum()}")

# Encode categorical variables: Gender and Geography
le = LabelEncoder()
df['Gender'] = le.fit_transform(df['Gender'])
df['Geography'] = le.fit_transform(df['Geography'])

print("\nEncoded Data Preview:")
print(df.head())

# 4. Train-Test Split & Standardization
X = df.drop(columns=['Exited'])
y = df['Exited']

# 80:20 Train-Test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Feature standardization
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\nTraining features shape: {X_train_scaled.shape}")
print(f"Testing features shape:  {X_test_scaled.shape}")

# 5. Build ANN Architecture
tf.random.set_seed(42)

model = Sequential([
    Input(shape=(X_train_scaled.shape[1],)),
    Dense(32, activation='relu'),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

# 6. Train ANN Model
history = model.fit(
    X_train_scaled, 
    y_train, 
    epochs=25, 
    batch_size=32, 
    validation_split=0.2, 
    verbose=1
)

# 7. Model Evaluation
y_pred_prob = model.predict(X_test_scaled)
y_pred = (y_pred_prob > 0.5).astype(int)

acc = accuracy_score(y_test, y_pred)
print(f"\nANN Test Accuracy: {acc:.4f} ({acc * 100:.2f}%)\n")
print("Classification Report:\n", classification_report(y_test, y_pred, target_names=['Stayed (0)', 'Exited (1)']))

# 8. Learning Curves & Confusion Matrix
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))

# 1. Training vs Validation Accuracy
axes[0].plot(history.history['accuracy'], label='Train Accuracy', color='#2980b9', lw=2)
axes[0].plot(history.history['val_accuracy'], label='Val Accuracy', color='#e67e22', lw=2)
axes[0].set_title('ANN Accuracy Curve', fontweight='bold')
axes[0].set_xlabel('Epochs')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, linestyle='--', alpha=0.6)

# 2. Training vs Validation Loss
axes[1].plot(history.history['loss'], label='Train Loss', color='#2980b9', lw=2)
axes[1].plot(history.history['val_loss'], label='Val Loss', color='#e74c3c', lw=2)
axes[1].set_title('ANN Loss Curve', fontweight='bold')
axes[1].set_xlabel('Epochs')
axes[1].set_ylabel('Binary Crossentropy Loss')
axes[1].legend()
axes[1].grid(True, linestyle='--', alpha=0.6)

# 3. Confusion Matrix Heatmap
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[2],
            xticklabels=['Stayed (0)', 'Exited (1)'], 
            yticklabels=['Stayed (0)', 'Exited (1)'])
axes[2].set_title(f'Confusion Matrix\n(Accuracy: {acc*100:.2f}%)', fontweight='bold')
axes[2].set_xlabel('Predicted Label')
axes[2].set_ylabel('True Label')

plt.tight_layout()
plt.show()

# ==============================================================================
# VIVA QUICK REFERENCE:
# 1. Activation Functions: ReLU for hidden layers (avoids vanishing gradient);
#    Sigmoid for output layer (maps output to probability [0, 1]).
# 2. Optimizer: Adam combines AdaGrad and RMSProp with adaptive learning rates.
# 3. Dropout: Regularization technique randomly deactivating neurons to prevent overfitting.
# ==============================================================================